In [7]:
# app py
from flask import Flask, render_template, request, session, redirect, url_for, make_response
import pandas as pd
import csv
import io
import os
app = Flask(__name__)
app.secret_key = 'eval_study_2026'


In [16]:
## configuration
import os

CSV_PATH = os.path.join(
    os.path.dirname("Assistive Technology/output/model_outputs.csv"),
    "data",
    "model_outputs.csv"
)

In [17]:
# The five rating dimensions
Dimensions = ['Clarity', 'Appropriateness', 'Accuracy', 'Consistency', 'Empathy']
# Evaluator expertise dropdown options
Expertise_options = [
    'Communication disablitis/AAC',
    'Cognitive / intellectual disabilties',
    'AI accessibility research',
    'Text simplification/ plain language',
    'Other',
]

In [28]:
# load csv data
def load_prompts():
    df = pd.read_csv(r"C:\Users\kehin\Desktop\Assistive Technology\outputs\model_outputs.csv")
    def clean(value):
        if pd.isna(value):
            return 'no output generated.'
        return str (value).strip()
        prompts=[]
        for _, row in df.iterrows():
            prompt.appemd({
                'id':         row['prompt_id'],
            'category':   row['category'],
            'disability': row['disability_type'],
            'task':       row['task_type'].replace('_', ' ').title(),
            'context':    clean(row['notes']),
            'prompt':     clean(row['prompt_text']),
            'model_a':    clean(row['gpt4_output']),    # blinded
            'model_b':    clean(row['claude_output']),  # blinded
            'model_c':    clean(row['groq_output']),    # blinded
        })
            return prompts
            

# Load once at startup - stored in memory
prompts= (r"C:\Users\kehin\Desktop\Assistive Technology\outputs\model_outputs.csv")
total = len(prompts)
print(f"loaded {total} prompt from CSV")
    

loaded 69 prompt from CSV


In [47]:
# ROUTE 1: Evaluator info screen
@app.route('/', methods=['get', 'post'])
def index():
    error = None
 
    if request.method == 'post':
        name        = request.form.get('name', '').strip()
        expertise   = request.form.get('expertise', '').strip()
        institution = request.form.get('institution', '').strip()
 
        if not name or not expertise:
            error = 'Please enter your name and select your area of expertise.'
        else:
            session.clear()
            session['evaluator'] = {
                'name':        name,
                'expertise':   expertise,
                'institution': institution,
            }
            session['current'] = 0    # which prompt we are on (0-indexed)
            session['ratings'] = {}   # stores all ratings keyed by prompt index
 
            return redirect(url_for('evaluate'))
 
    return render_template('index.html',
                           expertise_options=EXPERTISE_OPTIONS,
                           error=error)
 

In [51]:
import flask
# Remove old app if it exists (prevents duplicate route error in Jupyter)
if 'app' in dir():
    del app

app = Flask(__name__)
app.secret_key = 'eval_study_2026'

# Evaluation screen
@app.route('/evaluate', methods=['GET', 'POST'])
def evaluate():
    if 'evaluator' not in session:
        return redirect(url_for('index'))
 
    if request.method == 'POST':
        idx = session['current']
 
        # Save ratings from form
        # Form sends hidden inputs named a_0..a_4 (Model A), b_0..b_4, c_0..c_4
        session['ratings'][str(idx)] = {
            'model_a': [int(request.form.get(f'a_{d}', 0)) for d in range(5)],
            'model_b': [int(request.form.get(f'b_{d}', 0)) for d in range(5)],
            'model_c': [int(request.form.get(f'c_{d}', 0)) for d in range(5)],
            'notes':   request.form.get('notes', '').strip(),
        }
        session.modified = True
 
        action = request.form.get('action')
 
        if action == 'next' and idx < TOTAL - 1:
            session['current'] += 1
        elif action == 'back' and idx > 0:
            session['current'] -= 1
        elif action == 'submit':
            return redirect(url_for('done'))
 
        return redirect(url_for('evaluate'))
 
    # GET - show current prompt
    idx    = session['current']
    prompt = PROMPTS[idx]
    saved  = session['ratings'].get(str(idx), {})
 
    return render_template('evaluate.html',
        evaluator = session['evaluator'],
        prompt    = prompt,
        idx       = idx,
        total     = TOTAL,
        percent   = round((idx / TOTAL) * 100),
        saved     = saved,
        dims      = DIMENSIONS,
    )
 
 
# ROUTE 3: Completion screen
@app.route('/done')
def done():
    if 'evaluator' not in session:
        return redirect(url_for('index'))
 
    return render_template('done.html',
        evaluator = session['evaluator'],
        total     = TOTAL,
    )
 
 
# ROUTE 4: Download ratings as CSV
@app.route('/download')
def download():
    if 'evaluator' not in session:
        return redirect(url_for('index'))
 
    evaluator = session['evaluator']
    ratings   = session.get('ratings', {})
 
    # Build CSV in memory (no file saved to disk)
    output = io.StringIO()
    writer = csv.writer(output)
 
    # Header row
    writer.writerow([
        'evaluator_name', 'evaluator_expertise', 'evaluator_institution',
        'prompt_id', 'disability_type', 'task_type', 'context',
        'model_a_clarity', 'model_a_appropriateness', 'model_a_accuracy',
        'model_a_consistency', 'model_a_empathy',
        'model_b_clarity', 'model_b_appropriateness', 'model_b_accuracy',
        'model_b_consistency', 'model_b_empathy',
        'model_c_clarity', 'model_c_appropriateness', 'model_c_accuracy',
        'model_c_consistency', 'model_c_empathy',
        'notes',
    ])
 
    # One data row per prompt
    for i, prompt in enumerate(PROMPTS):
        r = ratings.get(str(i), {
            'model_a': [0, 0, 0, 0, 0],
            'model_b': [0, 0, 0, 0, 0],
            'model_c': [0, 0, 0, 0, 0],
            'notes':   '',
        })
        writer.writerow([
            evaluator['name'],
            evaluator['expertise'],
            evaluator['institution'],
            prompt['id'],
            prompt['disability'],
            prompt['task'],
            prompt['context'],
            *r['model_a'],   # unpacks 5 scores: clarity, approp, accuracy, consist, empathy
            *r['model_b'],
            *r['model_c'],
            r['notes'],
        ])
 
    # Send as downloadable file - named after the evaluator
    response = make_response(output.getvalue())
    filename = f"ratings_{evaluator['name'].replace(' ', '_')}.csv"
    response.headers['Content-Disposition'] = f'attachment; filename={filename}'
    response.headers['Content-Type'] = 'text/csv'
    return response

if __name__ == '__main__':
    app.run(debug=True, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\kehin\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
